# Synthetic Data Generation with Diffusion Models — Google Colab Runner

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AleksyPrawilow/synthetic-data/blob/main/colab.ipynb)

> **Zero Local Bloat:** Run all diffusion models, image generation, CNN fine-tuning, and evaluation on a Google Colab cloud GPU. Your local computer stays clean without downloading hundreds of megabytes of datasets or multi-gigabyte model weights.

### ⚡ Hardware Accelerator Requirement
Make sure GPU acceleration is enabled before running:
1. In the top menu, select **Runtime** ➔ **Change runtime type**.
2. Under **Hardware accelerator**, select **T4 GPU** (or L4 / A100).
3. Click **Save**.

## 0. Verify GPU & Environment
Check that a CUDA-capable GPU is allocated and ready.

In [ ]:
import sys
import torch

print(f"Python Version: {sys.version.split()[0]}")
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"Active GPU: {torch.cuda.get_device_name(0)}")
    !nvidia-smi
else:
    print("\n⚠️ WARNING: GPU is NOT active!")
    print("Please go to: Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU -> Save")


## 1. Storage & Repository Setup
Select how you want to work:
- **Option A (Ephemeral / Fast):** Clone to `/content/synthetic-data`. Fastest disk I/O, but remember to push to GitHub or download your results at the end of the session.
- **Option B (Persistent):** Mount Google Drive so your runs, checkpoints, and downloaded datasets persist across Colab disconnects.

In [ ]:
# Optional: Mount Google Drive for persistent storage
# Uncomment if you want to store your repository or cache on Google Drive
# from google.colab import drive
# drive.mount('/content/drive')


In [ ]:
import os
from pathlib import Path

# Repository to clone
REPO_URL = "https://github.com/AleksyPrawilow/synthetic-data.git"
REPO_DIR = Path("/content/synthetic-data")

if not REPO_DIR.exists():
    print(f"Cloning {REPO_URL}...")
    !git clone {REPO_URL} {REPO_DIR}
else:
    print(f"Repository exists at {REPO_DIR}.")

%cd {REPO_DIR}
print(f"Current directory: {Path.cwd()}")


## 2. Install Project Dependencies
Install the project in editable mode (`-e .`). This registers the course CLI commands in Colab:
- `synth-data`
- `synth-train`
- `synth-generate`
- `synth-test`

In [ ]:
# Install package dependencies and CLI tools into Colab environment
!pip install -e .


## 3. Download Data & Verify Starter Code
Download the Oxford-IIIT Pet dataset (~800 MB) into `data/` and verify the course split, then run tests to verify the installation.

In [ ]:
# Download Oxford-IIIT Pet dataset and check split
!synth-data


In [ ]:
# Run starter tests
!pytest


## 4. Colab Visual Inspection & Comparison Helpers
Run this cell to load helper functions for displaying contact sheets, inspecting seed metrics, and comparing runs directly in Colab cells.

In [ ]:
import json
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display

def show_contact_sheet(pool_name: str, width: int = 700):
    """Display the contact sheet of a pool inline in the notebook."""
    sheet_path = Path("pools") / pool_name / "contact-sheet.jpg"
    if sheet_path.exists():
        print(f"Contact sheet for: {pool_name}")
        display(Image(filename=str(sheet_path), width=width))
    else:
        print(f"⚠️ Contact sheet not found at: {sheet_path}")

def load_run_results(run_name: str, split: str = "val") -> pd.DataFrame:
    """Load results.json (val) or test-results.json (test) as a DataFrame."""
    fname = "test-results.json" if split == "test" else "results.json"
    path = Path("runs") / run_name / fname
    if not path.exists():
        print(f"Results file not found: {path}")
        return pd.DataFrame()
    with open(path) as f:
        data = json.load(f)
    rows = []
    for s in data.get("seeds", []):
        rows.append({
            "run": run_name,
            "seed": s["seed"],
            "egyptian_mau_recall": s["egyptian_mau_recall"],
            "macro_f1": s["macro_f1"],
            "accuracy": s.get("accuracy", None)
        })
    return pd.DataFrame(rows)

def compare_runs(run_names: list[str], split: str = "val") -> pd.DataFrame:
    """Display side-by-side metric comparison across runs."""
    records = []
    for name in run_names:
        df = load_run_results(name, split=split)
        if not df.empty:
            records.append({
                "Run": name,
                "Split": split,
                "Mau Recall (mean)": f"{df['egyptian_mau_recall'].mean():.4f}",
                "Mau Recall (std)": f"{df['egyptian_mau_recall'].std():.4f}",
                "Macro F1 (mean)": f"{df['macro_f1'].mean():.4f}",
                "Macro F1 (std)": f"{df['macro_f1'].std():.4f}",
                "Seeds": len(df)
            })
    if records:
        summary_df = pd.DataFrame(records)
        display(summary_df)
        return summary_df
    else:
        print("No run data found to compare.")
        return pd.DataFrame()

print("✅ Helper functions loaded: show_contact_sheet(), load_run_results(), compare_runs()")


## Meeting 1 — Real-Data Baselines
1. Implement `oversample()` and `augment()` in `src/synthdata/balance.py`.
2. Train the three baseline models (`real`, `oversample`, `augment`).
3. Compare on the validation set, select your candidate, then evaluate on the test photos.

In [ ]:
# Check src/synthdata/balance.py
!head -n 45 src/synthdata/balance.py


In [ ]:
# Train the 3 baselines (5 seeds each)
!synth-train --name m1-real --method real
!synth-train --name m1-oversample --method oversample
!synth-train --name m1-augment --method augment


In [ ]:
# Compare validation results
compare_runs(["m1-real", "m1-oversample", "m1-augment"], split="val");


In [ ]:
# Evaluate baselines on the test photos
!synth-test --run m1-real
!synth-test --run m1-oversample
!synth-test --run m1-augment

compare_runs(["m1-real", "m1-oversample", "m1-augment"], split="test");


## Meeting 2 — Text-to-Image Generation
Synthesize pools of Egyptian Mau images with Stable Diffusion 1.5 (or SDXL-Turbo), view the contact sheets, and train classifiers on the synthetic data.

In [ ]:
# Define your prompt
PROMPT = "a clear close-up portrait of an Egyptian Mau cat with spotted coat and green eyes, highly detailed photo"

# Generate base pool
!synth-generate --name m2-base --prompt "{PROMPT}"

# View contact sheet inline
show_contact_sheet("m2-base")


In [ ]:
# Train classifier with m2-base pool
!synth-train --name m2-base --method synthetic --pool m2-base

# Example parameter variation: Guidance scale
!synth-generate --name m2-guidance-3 --prompt "{PROMPT}" --guidance 3.0
!synth-train --name m2-guidance-3 --method synthetic --pool m2-guidance-3 --tag guidance=3.0

compare_runs(["m1-real", "m2-base", "m2-guidance-3"], split="val");


## Meeting 3 — Generation from Training Photos
Condition generation on the 15 real Egyptian Mau training photos:
- `--mode img2img`: `--strength` controls noise replacement.
- `--mode ip-adapter`: `--ip-adapter-scale` steers generation using image features.

In [ ]:
# img2img (example strength 0.5)
!synth-generate --name m3-strength-0.5 --mode img2img --strength 0.5 --prompt "{PROMPT}"
!synth-train --name m3-strength-0.5 --method synthetic --pool m3-strength-0.5 --tag mode=img2img --tag strength=0.5
show_contact_sheet("m3-strength-0.5")

# IP-Adapter (example scale 0.5)
!synth-generate --name m3-ip-0.5 --mode ip-adapter --ip-adapter-scale 0.5 --prompt "{PROMPT}"
!synth-train --name m3-ip-0.5 --method synthetic --pool m3-ip-0.5 --tag mode=ip-adapter --tag scale=0.5
show_contact_sheet("m3-ip-0.5")

compare_runs(["m1-real", "m3-strength-0.5", "m3-ip-0.5"], split="val");


## Meeting 4 — Editing Training Photos
- `--mode inpaint`: Keeps the cat and generates new backgrounds.
- `--mode outpaint`: Centers the cat and paints the expanded surroundings (`--outpaint-scale` 1.25 to 2.0).

In [ ]:
# Inpainting with background scenes
!synth-generate --name m4-backgrounds --mode inpaint \
    --prompt "an egyptian mau cat on a sunny wooden deck in a green garden" \
    --prompt "an egyptian mau cat sitting on a colorful oriental rug in a living room"
!synth-train --name m4-backgrounds --method synthetic --pool m4-backgrounds
show_contact_sheet("m4-backgrounds")

# Outpainting
!synth-generate --name m4-outpaint-1.5 --mode outpaint --outpaint-scale 1.5 --prompt "{PROMPT}"
!synth-train --name m4-outpaint-1.5 --method synthetic --pool m4-outpaint-1.5 --tag scale=1.5
show_contact_sheet("m4-outpaint-1.5")

compare_runs(["m1-real", "m4-backgrounds", "m4-outpaint-1.5"], split="val");


## Meeting 5 — Final Comparison & Evidence Generation
Collect the test results of your chosen runs from Meetings 1–4, generate summary plots, and save them in `evidence/`.

In [ ]:
# List of selected candidate runs from each meeting
SELECTED_RUNS = ["m1-real", "m1-oversample", "m1-augment"]  # Add your chosen m2, m3, m4 run names here

# Ensure test evaluation is run for all selected candidates
for run_name in SELECTED_RUNS:
    test_file = Path("runs") / run_name / "test-results.json"
    if not test_file.exists():
        print(f"Evaluating {run_name} on test photos...")
        !synth-test --run {run_name}

# Summary comparison
final_summary = compare_runs(SELECTED_RUNS, split="test")

# Generate comparison bar chart and save to evidence/
Path("evidence").mkdir(exist_ok=True)
plot_data = []
for r in SELECTED_RUNS:
    df = load_run_results(r, split="test")
    if not df.empty:
        plot_data.append({
            "Run": r,
            "Recall": df["egyptian_mau_recall"].mean(),
            "F1": df["macro_f1"].mean()
        })

if plot_data:
    pdf = pd.DataFrame(plot_data)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    ax1.bar(pdf["Run"], pdf["Recall"], color="#3182ce")
    ax1.set_title("Egyptian Mau Test Recall")
    ax1.set_ylim(0, 1.0)
    ax1.tick_params(axis='x', rotation=30)

    ax2.bar(pdf["Run"], pdf["F1"], color="#38a169")
    ax2.set_title("Macro F1 Test Score")
    ax2.set_ylim(0, 1.0)
    ax2.tick_params(axis='x', rotation=30)

    plt.tight_layout()
    evidence_plot = Path("evidence") / "meeting5_test_comparison.png"
    plt.savefig(evidence_plot, dpi=150)
    plt.show()
    print(f"Plot saved to: {evidence_plot}")


## 5. Save & Sync Your Work
Remember that Colab's default disk is erased when your session ends.
Save your work using either **Git Push to GitHub** or **One-Click Download**.

In [ ]:
# Option 1: Commit and push back to GitHub from Colab
import getpass

GIT_USER = "AleksyPrawilow"
GIT_EMAIL = "user@example.com"  # update with your email

!git config user.name "{GIT_USER}"
!git config user.email "{GIT_EMAIL}"

# Prompt for token (or set GITHUB_TOKEN in Colab Secrets)
try:
    from google.colab import userdata
    gh_token = userdata.get('GITHUB_TOKEN')
except Exception:
    gh_token = None

if not gh_token:
    gh_token = getpass.getpass("Enter GitHub Personal Access Token (leave empty to skip): ")

if gh_token:
    !git remote set-url origin https://{gh_token}@github.com/AleksyPrawilow/synthetic-data.git
    !git status
    # Track results, manifests, evidence, and experiment log
    !git add EXPERIMENTS.md evidence/ runs/*/results.json runs/*/test-results.json pools/*/manifest.jsonl pools/*/pool.json pools/*/contact-sheet.jpg
    !git commit -m "Colab run: update experiments and results" || echo "No changes to commit."
    !git push origin main
    print("✅ Successfully pushed to GitHub!")
else:
    print("Skipped git push.")


In [ ]:
# Option 2: Download results archive to your local computer
from google.colab import files

ZIP_FILE = "synthetic_data_results.zip"
# Exclude heavy model weights and full image directories (only keep contact sheets & manifests)
!zip -r -q {ZIP_FILE} evidence/ EXPERIMENTS.md runs/ pools/ \
    -x "runs/*/seed-*/model.pt" \
    -x "pools/*/images/*" \
    -x "*.tmp.png"

files.download(ZIP_FILE)
print(f"✅ Downloaded {ZIP_FILE} with all results, sheets, and evidence.")
